In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications.mobilenet import MobileNet, preprocess_input
import math

2025-07-07 10:56:06.592170: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-07 10:56:06.594583: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-07 10:56:06.623671: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-07 10:56:06.624110: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-07 10:56:07.273673: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

In [2]:
TRAIN_DATA_DIR = 'data/train/'
VALIDATION_DATA_DIR = 'data/val/'
TRAIN_SAMPLES = 500
VALIDATION_SAMPLES = 500
NUM_CLASSES = 2
IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 64

In [3]:
def model_maker():
    base_model = MobileNet(
        include_top=False,
        input_shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    for layer in base_model.layers[:]:
        layer.trainable = False
    input = Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    custom_model = base_model(input)
    custom_model = GlobalAveragePooling2D()(custom_model)
    custom_model = Dense(64, activation='relu')(custom_model)
    custom_model = Dropout(0.5)(custom_model)
    predictions = Dense(NUM_CLASSES, activation='softmax')(custom_model)
    return Model(inputs=input, outputs=predictions)

In [4]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [5]:
train_generator = train_datagen.flow_from_directory(TRAIN_DATA_DIR,
    target_size=(IMG_WIDTH,
    IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=12345,
    class_mode='categorical')
validation_generator = val_datagen.flow_from_directory(
    VALIDATION_DATA_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_mode='categorical')

Found 500 images belonging to 2 classes.
Found 500 images belonging to 2 classes.


In [6]:
model = model_maker()

17225924/17225924 [==============================] - 1s 0us/step


In [7]:
model.compile(loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(0.001),
    metrics=['acc'])

In [8]:
model.fit_generator(
    train_generator,
    steps_per_epoch=math.ceil(float(TRAIN_SAMPLES) / BATCH_SIZE),
    epochs=10,
    validation_data=validation_generator,
    validation_steps=math.ceil(float(VALIDATION_SAMPLES) / BATCH_SIZE))

/tmp/ipykernel_21/1178591529.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  model.fit_generator(


Epoch 1/10
8/8 [==============================] - 7s 797ms/step - loss: 0.6138 - acc: 0.7220 - val_loss: 0.1331 - val_acc: 0.9680
Epoch 2/10
8/8 [==============================] - 5s 690ms/step - loss: 0.1925 - acc: 0.9240 - val_loss: 0.1199 - val_acc: 0.9620
Epoch 3/10
8/8 [==============================] - 5s 704ms/step - loss: 0.1486 - acc: 0.9480 - val_loss: 0.0752 - val_acc: 0.9740
Epoch 4/10
8/8 [==============================] - 5s 679ms/step - loss: 0.1251 - acc: 0.9600 - val_loss: 0.0678 - val_acc: 0.9780
Epoch 5/10
8/8 [==============================] - 5s 684ms/step - loss: 0.0998 - acc: 0.9640 - val_loss: 0.0628 - val_acc: 0.9780
Epoch 6/10
8/8 [==============================] - 5s 693ms/step - loss: 0.0860 - acc: 0.9700 - val_loss: 0.0602 - val_acc: 0.9800
Epoch 7/10
8/8 [==============================] - 6s 731ms/step - loss: 0.0690 - acc: 0.9620 - val_loss: 0.0586 - val_acc: 0.9760
Epoch 8/10
8/8 [==============================] - 6s 709ms/step - loss: 0.0575 - acc: 0.97

In [9]:
model.save('model.h5')

/usr/local/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
